# SW07/HLT Python + JAX + NumPyro on Colab

This notebook runs the Python translation on the `Smets_Wouters_2007_HLT` model. It does **not** install, clone, or execute Julia on Colab.

Use `Runtime -> Change runtime type -> GPU` before running. Start with `PROFILE_MODE = "calibration"`; switch to `PROFILE_MODE = "long"` after the cells compile and the likelihood changes under parameter perturbations.

## What This Runs

- Clones only the Python translation repo into `/content`.
- Installs a CUDA-enabled JAX wheel before importing JAX.
- Downloads the SW07 model source as text, then parses it with the Python MacroModelling-style parser.
- Uses the bundled SW07 reference steady state and synthetic benchmark payload from the Python repo.
- Solves the first-order model with the Schur/QZ path, evaluates a JAX Kalman likelihood, and evaluates a NumPyro log density.
- Optionally runs a tiny NumPyro NUTS smoke test, disabled by default because the full SW07 gradient path is still not the robust benchmark target.

In [ ]:
REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "codex/colab-jax-gemini-profile"  # switch to "main" after these notebook changes are merged

SW07_SOURCE_URL = "https://raw.githubusercontent.com/matyasfarkas/SurrogateNN_Estimation.jl/main/models/Smets_Wouters_2007_HLT.jl"

# "calibration" uses the bundled 80-period SW07 payload. "long" simulates a 240-period SW07 sample in Python.
PROFILE_MODE = "calibration"  # "calibration" or "long"
LONG_PERIODS = 240
TIMING_REPS = 5

QME_ALGORITHM = "schur"  # matches the Julia-style Schur/QZ branch; use "doubling" only for experiments
PARAMETER_NAMES = ["cprobp", "cindp", "curvp"]
PRIOR_WIDTH_SCALE = 0.005
PRIOR_WIDTH_FLOOR = 1.0e-4

RUN_NUTS_SMOKE = False
NUTS_WARMUP = 4
NUTS_SAMPLES = 4

JAX_BACKEND = "auto"  # "auto", "cuda13", "cuda12", or "cpu"
STRICT_GPU = True
FORCE_REINSTALL_JAX = False  # leave False for normal two-pass Colab setup runs
UPDATE_EXISTING_CLONE = True
SHOCK_SEED = 20260720


In [ ]:
import os
import re
import sys
import subprocess
import time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

def run_text(cmd, cwd=None, check=False):
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    text = (proc.stdout or "") + (proc.stderr or "")
    print(text)
    return text

nvidia_smi = run_text(["bash", "-lc", "nvidia-smi || true"])
cuda_major = None
match = re.search(r"CUDA Version:\s*([0-9]+)", nvidia_smi)
if match:
    cuda_major = int(match.group(1))
driver_match = re.search(r"Driver Version:\s*([0-9.]+)", nvidia_smi)
driver_version = driver_match.group(1) if driver_match else "unknown"
print("Detected NVIDIA driver:", driver_version, "CUDA major:", cuda_major)

os.environ["JAX_ENABLE_X64"] = "1"
if JAX_BACKEND == "cpu":
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
else:
    os.environ.pop("JAX_PLATFORMS", None)
    os.environ.pop("JAX_PLATFORM_NAME", None)
    removed_ld_library_path = os.environ.pop("LD_LIBRARY_PATH", None)
    if removed_ld_library_path:
        print("Cleared LD_LIBRARY_PATH so JAX uses pip-installed CUDA libraries.")

ROOT = Path("/content/SurrogateNN_DSGE")
if not ROOT.exists():
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)])
elif UPDATE_EXISTING_CLONE:
    run(["git", "fetch", "origin", BRANCH], cwd=ROOT)
    run(["git", "checkout", BRANCH], cwd=ROOT)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=ROOT)
os.chdir(ROOT)
print("Python repo:", ROOT)

if JAX_BACKEND == "auto":
    if cuda_major is None:
        selected_jax_extra = "cuda13" if STRICT_GPU else "cpu"
    elif cuda_major >= 13:
        selected_jax_extra = "cuda13"
    else:
        selected_jax_extra = "cuda12"
else:
    selected_jax_extra = JAX_BACKEND
if selected_jax_extra not in {"cpu", "cuda12", "cuda13"}:
    raise ValueError(f"Unsupported JAX_BACKEND={selected_jax_extra!r}.")
print("Selected JAX install target:", selected_jax_extra)

COLAB_SETUP_VERSION = "sw07-cuda-jax-numpy-lt23-v2"
INSTALL_MARKER = Path(f"/content/.surrogatenn_dsge_{COLAB_SETUP_VERSION}_{selected_jax_extra}")
need_install = FORCE_REINSTALL_JAX or not INSTALL_MARKER.exists()
if need_install and "jax" in sys.modules:
    raise RuntimeError("JAX is already imported. Restart the Colab runtime before changing JAX wheels.")

if need_install:
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    run([
        sys.executable, "-m", "pip", "uninstall", "-y",
        "jax", "jaxlib", "jax-cuda12-pjrt", "jax-cuda12-plugin",
        "jax-cuda13-pjrt", "jax-cuda13-plugin",
    ], check=False)
    jax_package = "jax" if selected_jax_extra == "cpu" else f"jax[{selected_jax_extra}]"
    run([
        sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir",
        "numpy>=2.1,<2.3", jax_package,
    ])
    run([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "sympy>=1.13,<2", "scipy>=1.14,<1.16", "numpyro>=0.20", "pytest>=8.0,<9",
    ])
    run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
    run([sys.executable, "-m", "pip", "show", "jax", "jaxlib", "numpy", "numpyro", "scipy"], check=False)
    INSTALL_MARKER.write_text("installed\n")
    print("Colab wheels changed. Restarting the runtime once; rerun all cells after reconnect.")
    os.kill(os.getpid(), 9)
else:
    print("Colab setup marker found; skipping wheel installation:", INSTALL_MARKER)
    run([sys.executable, "-m", "pip", "show", "jax", "jaxlib", "numpy", "numpyro", "scipy"], check=False)

sys.path.insert(0, str(ROOT / "src"))

import json
import urllib.request
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

from surrogatenn_dsge import (
    assemble_parameter_vector,
    build_numpyro_kalman_model_jax,
    evaluate_numpyro_kalman_log_density_jax,
    kalman_loglikelihood_from_model_jax,
    parse_macro_model,
    rollout_first_order_solution,
    solve_first_order_model,
)

print("Python:", sys.version)
print("JAX:", jax.__version__)
print("NumPyro:", numpyro.__version__)
print("JAX x64 enabled:", jax.config.read("jax_enable_x64"))
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())
gpu_devices = [device for device in jax.devices() if device.platform in {"gpu", "cuda"}]
if selected_jax_extra != "cpu" and not gpu_devices:
    raise RuntimeError("CUDA JAX was requested, but JAX did not report a GPU device.")
if selected_jax_extra != "cpu":
    probe = (jnp.ones((256, 256), dtype=jnp.float32) @ jnp.ones((256, 256), dtype=jnp.float32)).block_until_ready()
    print("GPU probe result:", float(probe[0, 0]))


In [ ]:
payload_path = ROOT / "benchmarks" / "results" / "test_payloads.json"
payload = json.loads(payload_path.read_text())
case = next(entry for entry in payload["cases"] if entry["name"] == "medium_sw07_hlt")

model_source_path = ROOT / "benchmarks" / "results" / "Smets_Wouters_2007_HLT_colab.jl"
if not model_source_path.exists():
    print("Downloading SW07 model source text:", SW07_SOURCE_URL)
    model_source = urllib.request.urlopen(SW07_SOURCE_URL, timeout=60).read().decode("utf-8")
    model_source_path.write_text(model_source)
else:
    model_source = model_source_path.read_text()

model = parse_macro_model(model_source)
steady_state = np.asarray(case["reference_steady_state"], dtype=np.float64)
observables = tuple(case["observables"])
theta0 = np.asarray(model.parameter_values, dtype=np.float64)

print("Model:", case["model_symbol"])
print("Variables:", len(model.timings.var), "Shocks:", len(model.timings.exo), "Parameters:", len(model.parameter_names))
print("Observables:", observables)

t0 = time.perf_counter()
first_order = solve_first_order_model(
    model,
    steady_state=steady_state,
    qme_algorithm=QME_ALGORITHM,
)
solve_seconds = time.perf_counter() - t0
assert bool(first_order.solution.converged), "SW07 first-order solve did not converge."
print(f"First-order solve converged in {solve_seconds:.3f}s with QME algorithm {QME_ALGORITHM!r}.")

if PROFILE_MODE == "calibration":
    observations = np.asarray(case["observations"], dtype=np.float64)
    print("Using bundled benchmark observations:", observations.shape)
elif PROFILE_MODE == "long":
    shock_names = tuple(model.timings.exo)
    shock_sigmas = np.asarray([float(case["shock_sigmas"][name]) for name in shock_names], dtype=np.float64)
    rng = np.random.default_rng(SHOCK_SEED)
    shock_matrix = shock_sigmas[:, None] * rng.standard_normal((len(shock_names), LONG_PERIODS))
    full_levels = np.asarray(
        rollout_first_order_solution(first_order.solution.solution_matrix, model.timings, shock_matrix),
        dtype=np.float64,
    ) + steady_state[:, None]
    observation_indices = [model.timings.var.index(name) for name in observables]
    observations = full_levels[observation_indices, :]
    print("Generated Python-only long SW07 observations:", observations.shape)
else:
    raise ValueError("PROFILE_MODE must be 'calibration' or 'long'.")


In [ ]:
kalman_jit = jax.jit(
    lambda theta: kalman_loglikelihood_from_model_jax(
        model,
        observations,
        observables=observables,
        parameter_values=theta,
        steady_state=steady_state,
        qme_algorithm=QME_ALGORITHM,
        on_failure_loglikelihood=-1.0e12,
    )
)

t0 = time.perf_counter()
ll0 = kalman_jit(theta0).block_until_ready()
first_call_seconds = time.perf_counter() - t0

times = []
for _ in range(TIMING_REPS):
    t0 = time.perf_counter()
    kalman_jit(theta0).block_until_ready()
    times.append(time.perf_counter() - t0)

print("SW07 Kalman log likelihood:", float(ll0))
print("First JIT call seconds:", first_call_seconds)
print("Steady median seconds:", float(np.median(times)) if times else None)

theta_alt = theta0.copy()
for name, factor in {"cprobp": 0.995, "cindp": 1.005}.items():
    idx = model.parameter_names.index(name)
    theta_alt[idx] = theta_alt[idx] * factor
ll_alt = kalman_jit(theta_alt).block_until_ready()
print("Alternative-parameter log likelihood:", float(ll_alt))
assert not np.isclose(float(ll0), float(ll_alt), rtol=0.0, atol=1e-8), (
    "Likelihood did not change under SW07 parameter perturbations."
)


In [ ]:
priors = {}
parameter_samples = {}
for name in PARAMETER_NAMES:
    idx = model.parameter_names.index(name)
    center = float(theta0[idx])
    width = max(abs(center) * PRIOR_WIDTH_SCALE, PRIOR_WIDTH_FLOOR)
    lower = center - width
    upper = center + width
    if center > 0.0 and lower <= 0.0:
        lower = max(center * 0.5, np.finfo(float).tiny)
    priors[name] = dist.Uniform(lower, upper)
    parameter_samples[name] = jnp.asarray(center, dtype=jnp.float64)
    print(f"Prior {name}: Uniform({lower:.8g}, {upper:.8g}), sample={center:.8g}")

log_density_jit = jax.jit(
    lambda: evaluate_numpyro_kalman_log_density_jax(
        model,
        observations,
        priors,
        parameter_samples,
        observables=observables,
        steady_state=steady_state,
        base_parameter_values=theta0,
        qme_algorithm=QME_ALGORITHM,
        on_failure_loglikelihood=-1.0e12,
    )
)

t0 = time.perf_counter()
ld0 = log_density_jit().block_until_ready()
print("NumPyro/JAX SW07 log density:", float(ld0), "first call seconds:", time.perf_counter() - t0)

parameter_samples_alt = dict(parameter_samples)
parameter_samples_alt[PARAMETER_NAMES[0]] = jnp.asarray(float(parameter_samples[PARAMETER_NAMES[0]]) * 0.995, dtype=jnp.float64)
ld_alt = evaluate_numpyro_kalman_log_density_jax(
    model,
    observations,
    priors,
    parameter_samples_alt,
    observables=observables,
    steady_state=steady_state,
    base_parameter_values=theta0,
    qme_algorithm=QME_ALGORITHM,
    on_failure_loglikelihood=-1.0e12,
)
print("Alternative NumPyro/JAX SW07 log density:", float(ld_alt))
assert not np.isclose(float(ld0), float(ld_alt), rtol=0.0, atol=1e-8), (
    "NumPyro log density did not change under SW07 parameter perturbations."
)


In [ ]:
if RUN_NUTS_SMOKE:
    numpyro_model = build_numpyro_kalman_model_jax(
        model,
        observations,
        priors,
        observables=observables,
        steady_state=steady_state,
        base_parameter_values=theta0,
        qme_algorithm=QME_ALGORITHM,
        on_failure_loglikelihood=-1.0e12,
    )
    kernel = NUTS(numpyro_model, dense_mass=False, target_accept_prob=0.8)
    mcmc = MCMC(kernel, num_warmup=NUTS_WARMUP, num_samples=NUTS_SAMPLES, num_chains=1, progress_bar=True)
    mcmc.run(jax.random.PRNGKey(123))
    samples = mcmc.get_samples()
    print({name: np.asarray(value).tolist() for name, value in samples.items()})
else:
    print("NUTS smoke skipped. The SW07 NumPyro log density above is the robust default check.")
